In [12]:
import pandas as pd
from pathlib import Path
fp = "../data/raw/seattle-police-department-911-incident-response/Seattle_Police_Department_911_Incident_Response.csv"
df = pd.read_csv(fp)
df.head()

/tmp/ipykernel_201325/1927494154.py:4: DtypeWarning: Columns (0: CAD CDW ID) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fp)


,CAD CDW ID,CAD Event Number,General Offense Number,Event Clearance Code,Event Clearance Description,Event Clearance SubGroup,Event Clearance Group,Event Clearance Date,Hundred Block Location,District/Sector,Zone/Beat,Census Tract,Longitude,Latitude,Incident Location,Initial Type Description,Initial Type Subgroup,Initial Type Group,At Scene Time
0,﻿15736,10000246357,2010246357,242.0,FIGHT DISTURBANCE,DISTURBANCES,DISTURBANCES,07/17/2010 08:49:00 PM,3XX BLOCK OF PINE ST,M,M2,8100.2001,-122.338147,47.610975,"(47.610975163, -122.338146748)",NaN,NaN,NaN,NaN
1,15737,10000246471,2010246471,65.0,THEFT - MISCELLANEOUS,THEFT,OTHER PROPERTY,07/17/2010 08:50:00 PM,36XX BLOCK OF DISCOVERY PARK BLVD,Q,Q1,5700.1012,-122.404613,47.658325,"(47.658324899, -122.404612874)",NaN,NaN,NaN,NaN
2,15738,10000246255,2010246255,250.0,"MISCHIEF, NUISANCE COMPLAINTS","NUISANCE, MISCHIEF COMPLAINTS","NUISANCE, MISCHIEF",07/17/2010 08:55:00 PM,21XX BLOCK OF 3RD AVE,M,M2,7200.2025,-122.342843,47.613551,"(47.613551471, -122.342843234)",NaN,NaN,NaN,NaN
3,15739,10000246473,2010246473,460.0,TRAFFIC (MOVING) VIOLATION,TRAFFIC RELATED CALLS,TRAFFIC RELATED CALLS,07/17/2010 09:00:00 PM,7XX BLOCK OF ROY ST,D,D1,7200.1002,-122.341847,47.625401,"(47.625401388, -122.341846999)",NaN,NaN,NaN,NaN
4,15740,10000246330,2010246330,250.0,"MISCHIEF, NUISANCE COMPLAINTS","NUISANCE, MISCHIEF COMPLAINTS","NUISANCE, MISCHIEF",07/17/2010 09:00:00 PM,9XX BLOCK OF ALOHA ST,D,D1,6700.1009,-122.339709,47.627425,"(47.627424837, -122.339708605)",NaN,NaN,NaN,NaN


In [13]:
# Check the real column names present in the dataframe
print("Available columns:")
print(list(df.columns))

# Map the requested fields to the closest real columns in this dataset
requested_fields = {
    "final_call_type": ["Event Clearance Description", "Event Clearance Group", "Event Clearance SubGroup"],
    "cad_event_response_category": ["Event Clearance Group", "Initial Type Group"],
    "cad_event_original_time_queued": ["Event Clearance Date", "At Scene Time"],
    "cad_event_number": ["CAD Event Number"],
}

selected_columns = []
rename_map = {}

for target_name, candidate_names in requested_fields.items():
    actual_col = next((name for name in candidate_names if name in df.columns), None)
    if actual_col is not None:
        selected_columns.append(actual_col)
        rename_map[actual_col] = target_name

# Subset the dataframe with those real columns and rename them to the requested names
arrival_df = df[selected_columns].copy().rename(columns=rename_map)
arrival_df.head()

Available columns:
['CAD CDW ID', 'CAD Event Number', 'General Offense Number', 'Event Clearance Code', 'Event Clearance Description', 'Event Clearance SubGroup', 'Event Clearance Group', 'Event Clearance Date', 'Hundred Block Location', 'District/Sector', 'Zone/Beat', 'Census Tract', 'Longitude', 'Latitude', 'Incident Location', 'Initial Type Description', 'Initial Type Subgroup', 'Initial Type Group', 'At Scene Time']


,final_call_type,cad_event_response_category,cad_event_original_time_queued,cad_event_number
0,FIGHT DISTURBANCE,DISTURBANCES,07/17/2010 08:49:00 PM,10000246357
1,THEFT - MISCELLANEOUS,OTHER PROPERTY,07/17/2010 08:50:00 PM,10000246471
2,"MISCHIEF, NUISANCE COMPLAINTS","NUISANCE, MISCHIEF",07/17/2010 08:55:00 PM,10000246255
3,TRAFFIC (MOVING) VIOLATION,TRAFFIC RELATED CALLS,07/17/2010 09:00:00 PM,10000246473
4,"MISCHIEF, NUISANCE COMPLAINTS","NUISANCE, MISCHIEF",07/17/2010 09:00:00 PM,10000246330


In [14]:
junk_types = [
    'HANGUP', 'TELEPHONE HANGUP', 'EMERGENCY HANG UP',
    'TEST CALL', 'PRANK CALL', 'ADMINISTRATIVE',
    'ALARM TESTS', 'INFORMATION ONLY', 'HISTORY ONLY'
]

junk_norm = {t.upper() for t in junk_types}

arrival_df = arrival_df[
    arrival_df["final_call_type"]
    .fillna("")
    .astype(str)
    .str.upper()
    .apply(lambda x: x not in junk_norm)
].copy()

In [15]:
n_before = len(arrival_df)
n_dropped = arrival_df.isna().any(axis=1).sum()
drop_percent = round(n_dropped / n_before * 100, 2)

arrival_df = arrival_df.dropna().reset_index(drop=True)

print(f"Number of rows dropped due to missing values: {n_dropped}")
print(f"Percentage of dataset dropped: {drop_percent}%")
print(f"Remaining rows: {len(arrival_df)}")

Number of rows dropped due to missing values: 11584
Percentage of dataset dropped: 0.81%
Remaining rows: 1422269


In [16]:
target_year = 2016

arrival_df["cad_event_original_time_queued"] = pd.to_datetime(arrival_df["cad_event_original_time_queued"])
arrival_df = arrival_df[arrival_df["cad_event_original_time_queued"].dt.year == target_year].reset_index(drop=True)

print(f"Filtered to year {target_year}: {len(arrival_df)} rows remaining")

/tmp/ipykernel_201325/3070745003.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  arrival_df["cad_event_original_time_queued"] = pd.to_datetime(arrival_df["cad_event_original_time_queued"])


Filtered to year 2016: 254606 rows remaining


In [17]:
arrival_df

,final_call_type,cad_event_response_category,cad_event_original_time_queued,cad_event_number
0,"DISTURBANCE, OTHER",DISTURBANCES,2016-01-24 11:54:55,16000028163
1,SUSPICIOUS PERSON,SUSPICIOUS CIRCUMSTANCES,2016-01-24 11:57:35,16000028161
2,THEFT - MISCELLANEOUS,OTHER PROPERTY,2016-01-24 11:54:28,16000028159
3,ALACAD - COMMERCIAL BURGLARY (FALSE),FALSE ALACAD,2016-01-24 11:53:22,16000028134
4,TRESPASS,TRESPASS,2016-01-24 11:59:45,16000028114
...,...,...,...,...
254601,SHOPLIFT,SHOPLIFTING,2016-12-14 17:58:43,16000448856
254602,PARKS EXCLUSION,MISCELLANEOUS MISDEMEANORS,2016-12-17 08:58:25,16000452273
254603,MOTOR VEHICLE COLLISION,MOTOR VEHICLE COLLISION INVESTIGATION,2016-12-16 13:15:26,16000451186
254604,MOTOR VEHICLE COLLISION,MOTOR VEHICLE COLLISION INVESTIGATION,2016-12-14 16:28:56,16000448762


In [18]:
arrival_df["cad_event_original_time_queued"] = pd.to_datetime(arrival_df["cad_event_original_time_queued"])

n_unique_days = arrival_df["cad_event_original_time_queued"].dt.date.nunique()

min_ts = arrival_df["cad_event_original_time_queued"].min()
max_ts = arrival_df["cad_event_original_time_queued"].max()

delta_years = max_ts.year - min_ts.year
delta_months = delta_years * 12 + (max_ts.month - min_ts.month)

print(f"Number of unique days: {n_unique_days}")
print(f"Earliest timestamp: {min_ts}")
print(f"Latest timestamp: {max_ts}")
print(f"Number of months between min and max: {delta_months}")
print(f"Number of years between min and max: {delta_years}")

Number of unique days: 365
Earliest timestamp: 2016-01-01 00:00:46
Latest timestamp: 2016-12-31 23:55:37
Number of months between min and max: 11
Number of years between min and max: 0


In [19]:
missing_report = arrival_df.isna().sum().to_frame(name="missing_count")
missing_report["missing_percent"] = (missing_report["missing_count"] / len(arrival_df) * 100).round(2)
print("Data Quality Report - Missing Values")
print(missing_report)

Data Quality Report - Missing Values
                                missing_count  missing_percent
final_call_type                             0              0.0
cad_event_response_category                 0              0.0
cad_event_original_time_queued              0              0.0
cad_event_number                            0              0.0


In [20]:
missing_rows_df = arrival_df[arrival_df.isna().any(axis=1)].copy()
missing_rows_df

,final_call_type,cad_event_response_category,cad_event_original_time_queued,cad_event_number


In [21]:
from pathlib import Path

output_dir = Path.home() / "programming" / "arrival_analysis" / "data"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "seattle_911_call_arrivals.csv"
arrival_df.to_csv(output_path, index=False)

In [22]:
from pathlib import Path
import yaml
import pandas as pd

from data_prep.config import bootstrap_config, load_config
from data_prep.prepare import prepare_dataset

project_root = Path.cwd().resolve().parent
prepared_dir = project_root / "data" / "prepared" / "seattle-police-department-911-incident-response"
prepared_dir.mkdir(parents=True, exist_ok=True)

# Use the existing raw dictionary on disk and write the processed dictionary back into the staging directory.
dictionary_path = project_root / "data" / "raw" / "seattle-police-department-911-incident-response" / "data_dictionary_raw.csv"
raw_dd = pd.read_csv(dictionary_path)

config_path = bootstrap_config(
    config_dir=project_root / "configs",
    config_name="seattle-911-prep.yaml",
    dataset_name="seattle-police-department-911-incident-response",
    representation="tabular",
    task_characterization="time_series",
    staging_dir="./data/raw/seattle-police-department-911-incident-response",
    prepared_dataset_dir=str(prepared_dir),
    write=True,
)

target_year = 2016
prepared = df.copy()
requested_fields = {
    "final_call_type": ["Event Clearance Description", "Event Clearance Group", "Event Clearance SubGroup"],
    "cad_event_response_category": ["Event Clearance Group", "Initial Type Group"],
    "cad_event_original_time_queued": ["Event Clearance Date", "At Scene Time"],
    "cad_event_number": ["CAD Event Number"],
}

selected_columns = []
rename_map = {}
for target_name, candidate_names in requested_fields.items():
    actual_col = next((name for name in candidate_names if name in prepared.columns), None)
    if actual_col is not None:
        selected_columns.append(actual_col)
        rename_map[actual_col] = target_name

if not selected_columns:
    raise ValueError("No usable Seattle 911 call columns were found in the dataset")

prepared = prepared[selected_columns].copy().rename(columns=rename_map)
junk_types = {
    "HANGUP",
    "TELEPHONE HANGUP",
    "EMERGENCY HANG UP",
    "TEST CALL",
    "PRANK CALL",
    "ADMINISTRATIVE",
    "ALARM TESTS",
    "INFORMATION ONLY",
    "HISTORY ONLY",
}
prepared = prepared[
    prepared["final_call_type"].fillna("").astype(str).str.upper().apply(lambda x: x not in junk_types)
].copy()
prepared = prepared.dropna().reset_index(drop=True)
prepared["cad_event_original_time_queued"] = pd.to_datetime(
    prepared["cad_event_original_time_queued"], errors="coerce"
)
prepared = prepared[prepared["cad_event_original_time_queued"].dt.year == target_year].reset_index(drop=True)
processed_dd = raw_dd[raw_dd["attribute"].isin(prepared.columns)].copy().reset_index(drop=True)

prepare_result = prepare_dataset(
    dataset_name="seattle-police-department-911-incident-response",
    representation="tabular",
    task_characterization="time_series",
    staging_dir="./data/raw/seattle-police-department-911-incident-response",
    prepared_dataset_dir=prepared_dir,
    data=prepared,
    output_format="csv",
    filename="seattle-police-department-911-incident-response_filtered",
    metadata={
        "source": "seattle-police-department-911-incident-response",
        "preparation_logic": "filter_junk_calls_and_target_year",
        "target_year": target_year,
        "raw_data_dictionary": raw_dd,
        "data_dictionary": processed_dd,
    },
)

# Keep both dictionary files in the staging area; the processed dictionary is the one used for analysis.
config = load_config(config_path)
config["raw_data_dictionary_path"] = str(dictionary_path)
config["data_dictionary_path"] = str(Path(prepare_result["dictionary_files"]["data_dictionary"]))
config["processed_data_dictionary_path"] = config["data_dictionary_path"]
config["prepared_dataset_dir"] = str(prepared_dir)
with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(f"Bootstrap config: {config_path}")
print(f"Prepared dataset: {prepare_result['output_path']}")
print(f"Raw dictionary path: {config['raw_data_dictionary_path']}")
print(f"Processed dictionary path: {config['data_dictionary_path']}")
print(f"Dictionary files: {prepare_result.get('dictionary_files', {})}")
print(f"Files in prepared folder: {sorted(p.name for p in prepared_dir.iterdir())}")

pd.read_csv(prepare_result['output_path']).head()


/tmp/ipykernel_201325/1143979700.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  prepared["cad_event_original_time_queued"] = pd.to_datetime(


Bootstrap config: /home/rajiv/programming/kmds-dataset-util/configs/seattle-911-prep.yaml
Prepared dataset: /home/rajiv/programming/kmds-dataset-util/data/prepared/seattle-police-department-911-incident-response/seattle-police-department-911-incident-response_filtered.csv
Raw dictionary path: /home/rajiv/programming/kmds-dataset-util/data/raw/seattle-police-department-911-incident-response/data_dictionary_raw.csv
Processed dictionary path: data/raw/seattle-police-department-911-incident-response/data_dictionary.csv
Dictionary files: {'data_dictionary': 'data/raw/seattle-police-department-911-incident-response/data_dictionary.csv', 'raw_data_dictionary': 'data/raw/seattle-police-department-911-incident-response/data_dictionary_raw.csv'}
Files in prepared folder: ['data_dictionary.csv', 'data_dictionary_raw.csv', 'seattle-police-department-911-incident-response_filtered.csv']


,final_call_type,cad_event_response_category,cad_event_original_time_queued,cad_event_number
0,"DISTURBANCE, OTHER",DISTURBANCES,2016-01-24 11:54:55,16000028163
1,SUSPICIOUS PERSON,SUSPICIOUS CIRCUMSTANCES,2016-01-24 11:57:35,16000028161
2,THEFT - MISCELLANEOUS,OTHER PROPERTY,2016-01-24 11:54:28,16000028159
3,ALACAD - COMMERCIAL BURGLARY (FALSE),FALSE ALACAD,2016-01-24 11:53:22,16000028134
4,TRESPASS,TRESPASS,2016-01-24 11:59:45,16000028114
